In [ ]:
import os
import random
import shutil
from pathlib import Path

# Paths
gt_dir = Path("data\datasets\summer\ground_truth")
sentinel_dir = Path("data\datasets\summer\inputs")
output_dir = Path("data\datasets\summer\processed_dataset")

# Ratios
test_ratio = 0.2
val_ratio = 0.1

# Create folders (zonder outputs!)
for split in ["train", "test", "val"]:
    for subfolder in ["inputs", "ground_truth"]:
        os.makedirs(output_dir / split / subfolder, exist_ok=True)

# Verzamel alleen geldige paren
valid_pairs = []

for gt_image in gt_dir.glob("*.jpg"):
    sentinel_image = sentinel_dir / gt_image.name
    if sentinel_image.exists():
        valid_pairs.append((gt_image, sentinel_image))
    else:
        print(f"Missing input for {gt_image.name}")

print(f"Found {len(valid_pairs)} valid pairs")

# Shuffle
random.seed(42)
random.shuffle(valid_pairs)

# Split
test_count = int(len(valid_pairs) * test_ratio)
val_count = int(len(valid_pairs) * val_ratio)

val_pairs = valid_pairs[:val_count]
test_pairs = valid_pairs[val_count:val_count + test_count]
train_pairs = valid_pairs[val_count + test_count:]

splits = {
    "train": train_pairs,
    "test": test_pairs,
    "val": val_pairs
}

# Copy
for split, pairs in splits.items():
    print(f"processing {split} ({len(pairs)} images)")

    for gt_image, sentinel_image in pairs:
        try:
            shutil.copy(gt_image, output_dir / split / "ground_truth" / gt_image.name)
            shutil.copy(sentinel_image, output_dir / split / "inputs" / sentinel_image.name)

        except Exception as e:
            print(f"Error with {gt_image.name}: {e}")

print("Dataset ready!")